In [1]:
!pip install -q datasets transformers accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00


In [2]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
import pandas as pd

df = pd.read_csv("hf://datasets/BI55/MedText/medtext_2.csv")
print(df.columns)
print(df.iloc[0])

Index(['Prompt', 'Completion'], dtype='object')
Prompt        A 50-year-old male presents with a history of ...
Completion    This patient's history of recurrent kidney sto...
Name: 0, dtype: object


In [4]:
from datasets import Dataset

formatted_df = pd.DataFrame({
    "instruction": df["Prompt"],
    "input": "",
    "output": df["Completion"]
})

dataset = Dataset.from_pandas(formatted_df)
dataset = dataset.train_test_split(test_size=0.1)

dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1270
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 142
    })
})

In [5]:
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [6]:
def format_and_tokenize(example):
    full_prompt = f"Instruction: {example['instruction']}\nResponse:{example['output']}"
    return tokenizer(full_prompt, truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(format_and_tokenize, batched=False)

tokenized_dataset["train"][0]

Map:   0%|          | 0/1270 [00:00<?, ? examples/s]

Map:   0%|          | 0/142 [00:00<?, ? examples/s]

{'instruction': 'A 35-year-old female presents with a dull, constant headache and blurry vision that has been getting worse over the past two months. She has also noticed that her rings are fitting more tightly, and she has irregular menstrual periods. What is the likely diagnosis and recommended tests?',
 'input': '',
 'output': "Given the patient's symptoms, including a dull headache, visual changes, tighter fitting rings, and menstrual irregularity, the diagnosis of a pituitary adenoma, particularly a prolactinoma, should be considered. These tumors can cause headaches and visual symptoms due to mass effect and can also secrete prolactin, which could explain her menstrual irregularities and possible acral enlargement. The first step would be to obtain a serum prolactin level and an MRI of the brain with attention to the sella turcica to visualize the pituitary gland.",
 'input_ids': [1,
  2799,
  4080,
  29901,
  319,
  29871,
  29941,
  29945,
  29899,
  6360,
  29899,
  1025,
  12

In [7]:
from transformers import AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./tinyllama-medtext-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.604140
20,1.402265
30,1.273670
40,1.281614
50,1.282858
60,1.280571
70,1.259706
80,1.279002
90,1.255573
100,1.264791


TrainOutput(global_step=477, training_loss=1.214796230228192, metrics={'train_runtime': 868.2351, 'train_samples_per_second': 4.388, 'train_steps_per_second': 0.549, 'total_flos': 1.212144773234688e+16, 'train_loss': 1.214796230228192, 'epoch': 3.0})

In [10]:
model.save_pretrained("./tinyllama-medtext-lora")
tokenizer.save_pretrained("./tinyllama-medtext-lora")

('./tinyllama-medtext-lora/tokenizer_config.json',
 './tinyllama-medtext-lora/chat_template.jinja',
 './tinyllama-medtext-lora/tokenizer.json')

In [11]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="./tinyllama-medtext-lora",
    tokenizer="./tinyllama-medtext-lora",
    device=0
)

prompt = "Instruction: 65-year-old female presents with fatigue, weight loss, and elevated calcium levels.\nResponse:"

output = pipe(prompt, max_new_tokens=100, do_sample=True, temperature=0.7)[0]["generated_text"]

print(output)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Instruction: 65-year-old female presents with fatigue, weight loss, and elevated calcium levels.
Response:A 65-year-old female with a history of hypertension and diabetes, which often go uncontrolled, may have primary hyperparathyroidism. The elevation of calcium levels could be due to primary hyperparathyroidism, which can be refractory to therapy. She should be evaluated further with imaging tests, including CT or MRI of the thyroid, and possibly a bone density scan. Management could include bisph


In [12]:
!pip install -q evaluate bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.1 MB/s eta 0:00:00


In [14]:
from transformers import pipeline

pipe = pipeline("text-generation", model="./tinyllama-medtext-lora", tokenizer="./tinyllama-medtext-lora", device=0)

eval_inputs = dataset["test"]

eval_results = []

for example in eval_inputs:
    prompt = f"Instruction: {example['instruction']}\nResponse:"
    output = pipe(prompt, max_new_tokens=100, do_sample=False)[0]["generated_text"]

    predicted_response = output.split("Response:")[-1].strip()

    eval_results.append({
        "instruction": example["instruction"],
        "prediction": predicted_response,
        "reference": example["output"]
    })

eval_results[0]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

{'instruction': 'A 10-year-old child develops severe itching and redness around the mouth, throat discomfort and difficulty breathing after eating a shrimp salad. What is the likely diagnosis and immediate management?',
 'prediction': "This child's symptoms suggest a food allergy, likely to shellfish. The next step would be to confirm the diagnosis with a skin prick test or blood test, and to initiate appropriate management, including avoidance of the offending food and symptomatic treatment. If the child is allergic to shellfish, other foods that may cause symptoms include peanuts, tree nuts, and eggs. The child should be referred to a pediatric aller",
 'reference': "The child's symptoms suggest a severe allergic reaction, likely anaphylaxis, to shellfish. Immediate management should include administration of epinephrine, if available, and immediate transportation to an emergency department. Following the resolution of the acute episode, referral to an allergist for allergen confirma

In [15]:
!pip install rouge_score
!pip install bert_score

from evaluate import load

predictions = [ex["prediction"] for ex in eval_results]
references = [ex["reference"] for ex in eval_results]

rouge = load("rouge")
rouge_result = rouge.compute(predictions=predictions, references=references)

bertscore = load("bertscore")
bertscore_result = bertscore.compute(predictions=predictions, references=references, lang="en")

print("ROUGE Score:", rouge_result)
print("BERT Score:", bertscore_result)

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=daa59c37c1e755fb6b2b6ef438b8275d2aea9c707b0339d5f4ba5793c8e0370e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ROUGE Score: {'rouge1': np.float64(0.34634525741196087), 'rouge2': np.float64(0.10326521795241755), 'rougeL': np.float64(0.228292751537572), 'rougeLsum': np.float64(0.2297892282905362)}
BERT Score: {'precision': [0.8766102194786072, 0.8641491532325745, 0.8437708616256714, 0.8426395654678345, 0.8956805467605591, 0.8700791597366333, 0.8866873383522034, 0.8797364234924316, 0.8890820741653442, 0.881442666053772, 0.8987269401550293, 0.8692125678062439, 0.8996230959892273, 0.872454047203064, 0.874272882938385, 0.8525187969207764, 0.878014326095581, 0.8620767593383789, 0.8888139724731445, 0.9107872247695923, 0.8547685742378235, 0.8882197737693787, 0.8605109453201294, 0.8829145431518555, 0.8796818852424622, 0.86989426612854, 0.865432620048523, 0.8673076629638672, 0.877342164516449, 0.8814001679420471, 0.8758125305175781, 0.9053082466125488, 0.8706218600273132, 0.8984055519104004, 0.8847582340240479, 0.858761191368103, 0.8920662999153137, 0.8871503472328186, 0.8774592280387878, 0.90695673227310

In [16]:
from google.colab import files
model.save_pretrained("tinyllama-medtext-lora")
tokenizer.save_pretrained("tinyllama-medtext-lora")

('tinyllama-medtext-lora/tokenizer_config.json',
 'tinyllama-medtext-lora/chat_template.jinja',
 'tinyllama-medtext-lora/tokenizer.json')

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

adapter_path = "/content/tinyllama-medtext-lora"
model = PeftModel.from_pretrained(base_model, adapter_path)

model = model.merge_and_unload()

output_path = "/content/tinyllama-custom"
model.save_pretrained(output_path)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.save_pretrained(output_path)

print(f"Merged model saved to: {output_path}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/tinyllama-custom


In [18]:
import shutil
shutil.make_archive("tinyllama-custom", "zip", "tinyllama-custom")
from google.colab import files
files.download("tinyllama-custom.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import time
import numpy as np
from torch.profiler import profile, record_function, ProfilerActivity

TEST_PROMPTS = [
    "What are the symptoms of type 2 diabetes?",
    "How is hypertension diagnosed and treated?",
    "What is the difference between viral and bacterial pneumonia?",
    "Describe the mechanism of action of beta blockers.",
    "What are the risk factors for myocardial infarction?"
]

def benchmark(model, tokenizer, device, label, n_warmup=5, n_runs=30):
    model = model.to(device)
    model.eval()
    results = []

    for prompt in TEST_PROMPTS:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            max_length=128,
            truncation=True,
            padding=True
        ).to(device)

        # Warmup
        with torch.no_grad():
            for _ in range(n_warmup):
                model.generate(**inputs, max_new_tokens=100, do_sample=False)

        # Timed runs
        times = []
        with torch.no_grad():
            for _ in range(n_runs):
                if device == "cuda":
                    torch.cuda.synchronize()
                start = time.perf_counter()
                model.generate(**inputs, max_new_tokens=100, do_sample=False)
                if device == "cuda":
                    torch.cuda.synchronize()
                end = time.perf_counter()
                times.append((end - start) * 1000)

        results.append({
            "mean_ms": np.mean(times),
            "p50_ms": np.percentile(times, 50),
            "p95_ms": np.percentile(times, 95),
        })

    mean = np.mean([r["mean_ms"] for r in results])
    p50 = np.percentile([r["p50_ms"] for r in results], 50)
    p95 = np.percentile([r["p95_ms"] for r in results], 95)

    print(f"\n{'='*50}")
    print(f"Target: {label}")
    print(f"Mean: {mean:.2f}ms | P50: {p50:.2f}ms | P95: {p95:.2f}ms")
    return {"label": label, "mean_ms": mean, "p50_ms": p50, "p95_ms": p95}

all_results = {}

# 1. CPU baseline
print("[1/3] CPU baseline...")
cpu_model = AutoModelForCausalLM.from_pretrained(
    "/content/tinyllama-custom",
    torch_dtype=torch.float32
)
all_results["cpu"] = benchmark(cpu_model, tokenizer, "cpu", "CPU Baseline")
del cpu_model
torch.cuda.empty_cache()

# 2. GPU (T4 on Colab)
print("\n[2/3] GPU (T4)...")
gpu_model = AutoModelForCausalLM.from_pretrained(
    "/content/tinyllama-custom",
    torch_dtype=torch.float16
)
all_results["gpu"] = benchmark(gpu_model, tokenizer, "cuda", "GPU (T4)")

# 3. torch.compile()
print("\n[3/3] torch.compile() - TorchDynamo/TorchInductor...")
print("First run will be slow - compiling...")
compiled_model = torch.compile(gpu_model, mode="reduce-overhead")
all_results["compiled"] = benchmark(compiled_model, tokenizer, "cuda", "GPU + torch.compile()")
del gpu_model
torch.cuda.empty_cache()

# Summary
print("\n" + "="*60)
print("FINAL BENCHMARK RESULTS")
print("="*60)
baseline = all_results["cpu"]["mean_ms"]
for key, r in all_results.items():
    speedup = baseline / r["mean_ms"]
    print(f"{r['label']:<35} Mean: {r['mean_ms']:.2f}ms  ({speedup:.2f}x vs CPU)")

import json
with open("benchmark_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nSaved to benchmark_results.json")

[1/3] CPU baseline...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging


Target: CPU Baseline
Mean: 16609.45ms | P50: 1593.84ms | P95: 39640.66ms

[2/3] GPU (T4)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging


Target: GPU (T4)
Mean: 1363.65ms | P50: 39.90ms | P95: 3844.57ms

[3/3] torch.compile() - TorchDynamo/TorchInductor...
First run will be slow - compiling...


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging


Target: GPU + torch.compile()
Mean: 1380.81ms | P50: 40.83ms | P95: 3921.75ms

FINAL BENCHMARK RESULTS
CPU Baseline                        Mean: 16609.45ms  (1.00x vs CPU)
GPU (T4)                            Mean: 1363.65ms  (12.18x vs CPU)
GPU + torch.compile()               Mean: 1380.81ms  (12.03x vs CPU)

Saved to benchmark_results.json


In [5]:
print("\nRunning operator-level profiler...")
profile_model = AutoModelForCausalLM.from_pretrained(
    "/content/tinyllama-custom",
    torch_dtype=torch.float16
).to("cuda")

sample_input = tokenizer(
    TEST_PROMPTS[0],
    return_tensors="pt",
    max_length=128,
    truncation=True
).to("cuda")

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
    with_flops=True
) as prof:
    with record_function("biomedical_tinyllama_inference"):
        with torch.no_grad():
            profile_model.generate(
                **sample_input,
                max_new_tokens=50,
                do_sample=False
            )

print("\nTop 15 operators by CUDA time:")
print(prof.key_averages().table(
    sort_by="cuda_time_total",
    row_limit=15
))

prof.export_chrome_trace("tinyllama_trace.json")
print("Chrome trace saved")
del profile_model


Running operator-level profiler...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Top 15 operators by CUDA time:
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  Total MFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                         biomedical_tinyllama_inference         0.00%       0.000us         0.00%       0.000us       0.000us      84.932ms       428.96%      84.

In [6]:
from google.colab import files
import zipfile
import os

files.download("benchmark_results.json")
files.download("tinyllama_trace.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads started - check your browser
